# TFMv0 - CORN Ordinal Depression Severity Classification

This notebook contains the final CORN-based version of the multimodal depression severity pipeline before the project was reformulated as PHQ-8 score regression.

The task is framed as five-class ordinal classification using Conditional Ordinal Regression for Neural networks (CORN). The models combine facial graph representations with acoustic features and are evaluated primarily at participant level to reduce window-level bias.

> Before running this notebook, update the dataset paths in the configuration cell to match your local environment.


In [ ]:
# ============================================================
# ENVIRONMENT & REPRODUCIBILITY
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ---Standard library ---
import io
import time
import random
import zipfile
import warnings
from collections import Counter, defaultdict

# --- Data & numerics ---
import numpy as np
import pandas as pd

# --- PyTorch core ---------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf

# --- PyTorch Geometric ---
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, SAGEConv, global_mean_pool

# --- Transformers ---------
from transformers import Wav2Vec2Model, Wav2Vec2Processor

# --- PyTorch Lightning ---
import pytorch_lightning as pl
from pytorch_lightning import seed_everything
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.loggers import CSVLogger

# --- Data handling ---------
from torch.utils.data import Dataset, DataLoader
import requests
from bs4 import BeautifulSoup

# --- Scikit-learn ---------
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# --- Visualization ---------
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, clear_output
from tqdm import tqdm

# --- Warnings ---------------
warnings.filterwarnings("ignore", module="rich")
warnings.filterwarnings("ignore", message="In 2.9, this function")
warnings.filterwarnings("ignore", message='install "ipywidgets"')

# --- Reproducibility ------
SEED = 42
seed_everything(SEED, workers=True)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# --- Environment check ---
print("[OK] TFM environment initialized")
print(f"PyTorch     : {torch.__version__}")
print(f"Lightning   : {pl.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"VRAM used   : {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")


## 1. Environment and Reproducibility

This section imports the core libraries, sets the GPU device, and fixes random seeds for reproducibility.


In [ ]:
# --- Global configuration ------------------------------------------------------
BASE_DIR         = "/home/arosario/depression-gnn-project/DAIC_WOZ_features"
LABELS_PATH      = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/train_split_Depression_AVEC2017.csv"
SAVE_DIR         = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/DAIC_WOZ_graphs"
WAV2VEC_SAVE_DIR = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/preprocessed_wav2vec"
GRAPHS_DIR       = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/DAIC_WOZ_graphs"
WAV2VEC_DIR      = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/preprocessed_wav2vec"
CKPT_DIR         = "/home/arosario/depression-gnn-project/ckpts"
LOG_DIR          = "/home/arosario/depression-gnn-project/logs"
TMP_DIR          = "/home/arosario/depression-gnn-project/tmp"

WINDOW_SIZE   = 150   # frames COVAREP por ventana de agregacin
WINDOW_FRAMES = 10     # frames CLNF por ventana del modelo
STRIDE_FRAMES = 10     # stride entre ventanas
BATCH_SIZE    = 32
SAMPLE_RATE   = 16000
WINDOW_MS     = 300

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = {
    0: "Sin sintomas",
    1: "Leve",
    2: "Moderado",
    3: "Moderadamente severo",
    4: "Severo"
}

# --- Create dirs ------------
for d in [SAVE_DIR, GRAPHS_DIR, WAV2VEC_DIR, CKPT_DIR, LOG_DIR, TMP_DIR]:
    os.makedirs(d, exist_ok=True)

print("[OK] Global config initialized")
print(f"Device        : {DEVICE}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Window frames : {WINDOW_FRAMES}")
print(f"Stride frames : {STRIDE_FRAMES}")


## 2. Global Configuration

Project paths, feature directories, checkpoint directories, logging directories, training constants, and PHQ-8 class names are defined here.


In [ ]:
base_url = "https://dcapswoz.ict.usc.edu/wwwdaicwoz/"

output_dir = "depression-gnn-project/DAIC_WOZ_features/CLNF_COVAREP"
os.makedirs(output_dir, exist_ok=True)

# ---- IDs ----
allowed_ids = {
    # DEV IDs
    302, 307, 331, 335, 346, 367, 377, 381, 382, 388, 389, 390, 395,
    403, 404, 406, 413, 417, 418, 420, 422, 436, 439, 440, 451, 458,
    472, 476, 477, 482, 483, 484, 489, 490, 492,
    # Train IDs
    303, 304, 305, 310, 312, 313, 315, 316, 317, 318, 319, 320, 321,
    322, 324, 325, 326, 327, 328, 330, 333, 336, 338, 339, 340, 341,
    343, 344, 345, 347, 348, 350, 351, 352, 353, 355, 356, 357, 358,
    360, 362, 363, 364, 366, 368, 369, 370, 371, 372, 374, 375, 376,
    379, 380, 383, 385, 386, 391, 392, 393, 397, 400, 401, 402, 409,
    412, 414, 415, 416, 419, 423, 425, 426, 427, 428, 429, 430, 433,
    434, 437, 441, 443, 444, 445, 446, 447, 448, 449, 454, 455, 456,
    457, 459, 463, 464, 468, 471, 473, 474, 475, 478, 479, 485, 486,
    487, 488, 491,
}

# ---- Archivos a extraer ----
target_endings = (
    "COVAREP.csv",
    "CLNF_features3D.txt",
    "AU.csv"
)

# Fetch directory HTML
response = requests.get(base_url)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")
all_links = soup.find_all("a")

# Filtrar ZIPs válidos
zip_files = []
for link in all_links:
    href = link.get("href")
    if href and href.endswith("_P.zip"):
        try:
            pid = int(href.split("_")[0])
        except ValueError:
            continue

        if pid in allowed_ids:
            zip_files.append(href)

zip_files = sorted(set(zip_files))
print(f"Found {len(zip_files)} ZIPs matching allowed IDs.")

# Procesar ZIPs
for zip_name in zip_files:
    zip_url = base_url + zip_name
    pid = zip_name.replace(".zip", "")
    print(f"==== {pid} ====")

    try:
        r = requests.get(zip_url)
        r.raise_for_status()
    except Exception as e:
        print("Download failed:", e)
        continue

    zip_bytes = io.BytesIO(r.content)

    try:
        with zipfile.ZipFile(zip_bytes) as z:
            part_out = os.path.join(output_dir, pid)
            os.makedirs(part_out, exist_ok=True)

            for info in z.infolist():
                if info.filename.endswith(target_endings):
                    print("  extracting:", info.filename)
                    with z.open(info) as fsrc, open(
                        os.path.join(part_out, os.path.basename(info.filename)), "wb"
                    ) as fdst:
                        fdst.write(fsrc.read())

    except Exception as e:
        print("Extraction error:", e)

    time.sleep(1)

print("All done!")


## 3. Raw Feature Download and Audio Embedding Extraction

These cells download or prepare DAIC-WOZ feature files and Wav2Vec embeddings. They only need to be rerun when rebuilding the preprocessed feature cache.


In [ ]:
# ============================================================
# WAV2VEC BLOCK: Descarga .wav  extrae embedding  guarda en Drive
# ============================================================


os.makedirs(WAV2VEC_SAVE_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

base_url    = "https://dcapswoz.ict.usc.edu/wwwdaicwoz/"
SAMPLE_RATE = 16000
WINDOW_MS   = 300
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# ---- IDs ----
allowed_ids = {
    # DEV IDs
    302, 307, 331, 335, 346, 367, 377, 381, 382, 388, 389, 390, 395,
    403, 404, 406, 413, 417, 418, 420, 422, 436, 439, 440, 451, 458,
    472, 476, 477, 482, 483, 484, 489, 490, 492,
    # Train IDs
    303, 304, 305, 310, 312, 313, 315, 316, 317, 318, 319, 320, 321,
    322, 324, 325, 326, 327, 328, 330, 333, 336, 338, 339, 340, 341,
    343, 344, 345, 347, 348, 350, 351, 352, 353, 355, 356, 357, 358,
    360, 362, 363, 364, 366, 368, 369, 370, 371, 372, 374, 375, 376,
    379, 380, 383, 385, 386, 391, 392, 393, 397, 400, 401, 402, 409,
    412, 414, 415, 416, 419, 423, 425, 426, 427, 428, 429, 430, 433,
    434, 437, 441, 443, 444, 445, 446, 447, 448, 449, 454, 455, 456,
    457, 459, 463, 464, 468, 471, 473, 474, 475, 478, 479, 485, 486,
    487, 488, 491,
}

# Cargar wav2vec2 frozen
print("Cargando wav2vec2-base...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec2  = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(DEVICE)
wav2vec2.eval()
for p in wav2vec2.parameters():
    p.requires_grad = False
print(f"Listo  device: {DEVICE}")

# --- Funcin de extraccin ---------------------------------------------------
def extract_embeddings(wav_path, processor, model, device,
                       window_ms=300, sample_rate=16000):
    # Usar soundfile directamente para evitar problemas de backend
    audio_np, sr = sf.read(wav_path, dtype="float32")
    
    # Si es estreo  mono
    if audio_np.ndim > 1:
        audio_np = audio_np.mean(axis=1)
    
    # Resamplear si es necesario
    waveform = torch.tensor(audio_np)
    if sr != sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, sample_rate)
    
    window_samples = int(sample_rate * window_ms / 1000)
    n_windows      = len(waveform) // window_samples

    embeddings, timestamps = [], []

    for i in range(n_windows):
        chunk  = waveform[i*window_samples:(i+1)*window_samples].numpy()
        inputs = processor(chunk, sampling_rate=sample_rate,
                           return_tensors="pt", padding=True)
        with torch.no_grad():
            out = wav2vec2(inputs.input_values.to(device))
            emb = out.last_hidden_state.mean(dim=1).squeeze(0)
        embeddings.append(emb.cpu())
        timestamps.append(torch.tensor(i * window_ms / 1000.0))

    if not embeddings:
        return None, None
    return torch.stack(embeddings), torch.stack(timestamps)

# --- Fetch HTML ---------
soup      = BeautifulSoup(requests.get(base_url).text, "html.parser")
zip_files = sorted(set(
    l.get("href") for l in soup.find_all("a")
    if l.get("href", "").endswith("_P.zip")
    and int(l.get("href").split("_")[0]) in allowed_ids
))
print(f"ZIPs encontrados: {len(zip_files)}")

# --- Loop principal ---
skipped, processed, failed = [], [], []

for zip_name in zip_files:
    pid       = zip_name.replace(".zip", "")
    save_path = os.path.join(WAV2VEC_SAVE_DIR, f"{pid}.pt")

    # Ya procesado  saltar
    if os.path.exists(save_path):
        print(f"[SKIP] {pid}  embedding ya existe")
        skipped.append(pid)
        continue

    print(f"\n--- {pid} ---")

    # 1. Descargar ZIP a /tmp (disco Colab)
    try:
        r = requests.get(base_url + zip_name, timeout=120)
        r.raise_for_status()
    except Exception as e:
        print(f"  [ERROR] Descarga: {e}")
        failed.append(pid)
        continue

    # 2. Extraer solo el .wav a /tmp
    wav_path = None
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            for info in z.infolist():
                if info.filename.endswith(".wav"):
                    wav_path = os.path.join(TMP_DIR, os.path.basename(info.filename))
                    with z.open(info) as fsrc, open(wav_path, "wb") as fdst:
                        fdst.write(fsrc.read())
                    print(f"  .wav  {wav_path} ({os.path.getsize(wav_path)/1024**2:.1f} MB)")
                    break
    except Exception as e:
        print(f"  [ERROR] Extraccin: {e}")
        failed.append(pid)
        continue

    if wav_path is None:
        print(f"  [WARN] Sin .wav en ZIP")
        failed.append(pid)
        continue

    # 3. Extraer embeddings
    try:
        embeddings, timestamps = extract_embeddings(
            wav_path, processor, wav2vec2, DEVICE,
            window_ms=WINDOW_MS, sample_rate=SAMPLE_RATE
        )
        if embeddings is None:
            print(f"  [WARN] Audio demasiado corto")
            failed.append(pid)
        else:
            # 4. Guardar embedding en Drive
            torch.save({"embeddings": embeddings, "timestamps": timestamps}, save_path)
            print(f"  [OK] {embeddings.shape}  {save_path}")
            print(f"       Drive: {os.path.getsize(save_path)/1024**2:.1f} MB")
            processed.append(pid)
    except Exception as e:
        print(f"  [ERROR] Extraccin embedding: {e}")
        failed.append(pid)

    # 5. Borrar .wav de /tmp (liberar disco Colab)
    if wav_path and os.path.exists(wav_path):
        os.remove(wav_path)
        print(f"  .wav eliminado de /tmp")

    time.sleep(1)

# --- Resumen ---------------
print(f"\n{'---'*45}")
print(f"Procesados : {len(processed)}")
print(f"Saltados   : {len(skipped)}")
print(f"Fallidos   : {len(failed)}")
if failed:
    print(f"  {failed}")


## 4. Graph Cache Construction

This section builds intermediate `.pt` files containing facial graph data, acoustic features, and participant labels.


In [ ]:

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(TMP_DIR,  exist_ok=True)


# --- IDs ------------------------
allowed_ids = {
    # DEV IDs
    302, 307, 331, 335, 346, 367, 377, 381, 382, 388, 389, 390, 395,
    403, 404, 406, 413, 417, 418, 420, 422, 436, 439, 440, 451, 458,
    472, 476, 477, 482, 483, 484, 489, 490, 492,
    # Train IDs
    303, 304, 305, 310, 312, 313, 315, 316, 317, 318, 319, 320, 321,
    322, 324, 325, 326, 327, 328, 330, 333, 336, 338, 339, 340, 341,
    343, 344, 345, 347, 348, 350, 351, 352, 353, 355, 356, 357, 358,
    360, 362, 363, 364, 366, 368, 369, 370, 371, 372, 374, 375, 376,
    379, 380, 383, 385, 386, 391, 392, 393, 397, 400, 401, 402, 409,
    412, 414, 415, 416, 419, 423, 425, 426, 427, 428, 429, 430, 433,
    434, 437, 441, 443, 444, 445, 446, 447, 448, 449, 454, 455, 456,
    457, 459, 463, 464, 468, 471, 473, 474, 475, 478, 479, 485, 486,
    487, 488, 491,
}

# --- Facial topology ------
FACIAL_EDGES = [
    (0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),
    (8,9),(9,10),(10,11),(11,12),(12,13),(13,14),(14,15),(15,16),
    (17,18),(18,19),(19,20),(20,21),
    (22,23),(23,24),(24,25),(25,26),
    (27,28),(28,29),(29,30),
    (30,31),(31,32),(32,33),(33,34),(34,35),
    (36,37),(37,38),(38,39),(39,40),(40,41),(41,36),
    (42,43),(43,44),(44,45),(45,46),(46,47),(47,42),
    (48,49),(49,50),(50,51),(51,52),(52,53),(53,54),
    (54,55),(55,56),(56,57),(57,58),(58,59),(59,48),
]
edges_both = FACIAL_EDGES + [(b, a) for a, b in FACIAL_EDGES]
edge_index = torch.tensor(edges_both, dtype=torch.long).t().contiguous()

# --- Labels ------------------
def phq8_to_class(score):
    if score <= 4:    return 0
    elif score <= 9:  return 1
    elif score <= 14: return 2
    elif score <= 19: return 3
    else:             return 4

df_labels = pd.read_csv(LABELS_PATH, sep=",")
df_labels.columns = df_labels.columns.str.strip()
df_labels["PHQ8_Class"] = df_labels["PHQ8_Score"].apply(phq8_to_class)
df_labels["pid"] = df_labels["Participant_ID"].astype(str) + "_P"
df_labels = df_labels.set_index("pid")

# --- COVAREP columns ------
COVAREP_COLUMNS = [
    "F0", "VUV", "NAQ", "QOQ", "H1H2", "PSP", "MDQ",
    "peakSlope", "Rd", "Rd_conf",
    "MCEP_0","MCEP_1","MCEP_2","MCEP_3","MCEP_4",
    "MCEP_5","MCEP_6","MCEP_7","MCEP_8","MCEP_9",
    "MCEP_10","MCEP_11","MCEP_12","MCEP_13","MCEP_14",
    "MCEP_15","MCEP_16","MCEP_17","MCEP_18","MCEP_19",
    "MCEP_20","MCEP_21","MCEP_22","MCEP_23","MCEP_24",
    "MCEP_25","MCEP_26","MCEP_27","MCEP_28","MCEP_29",
    "MCEP_30","MCEP_31","MCEP_32","MCEP_33","MCEP_34",
    "MCEP_35","MCEP_36","MCEP_37","MCEP_38","MCEP_39",
    "MCEP_40","MCEP_41","MCEP_42","MCEP_43","MCEP_44",
    "MCEP_45","MCEP_46","MCEP_47","MCEP_48","MCEP_49",
    "HMPDM_0","HMPDM_1","HMPDM_2","HMPDM_3","HMPDM_4",
    "HMPDM_5","HMPDM_6","HMPDM_7","HMPDM_8","HMPDM_9",
    "HMPDD_0","HMPDD_1","HMPDD_2","HMPDD_3",
]
assert len(COVAREP_COLUMNS) == 74

# --- Processing functions ---------------------------------------------------------
def load_clnf_3d(path):
    df = pd.read_csv(path, sep=",", header=0, low_memory=False)
    df.columns = df.columns.str.strip()
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    return df

def load_covarep(path):
    df = pd.read_csv(path, header=None)
    if df.shape[1] == 74:
        df.columns = COVAREP_COLUMNS
    return df

def aggregate_covarep(df_audio, window_size=30):
    n_windows = len(df_audio) // window_size
    return pd.DataFrame([
        df_audio.iloc[i*window_size:(i+1)*window_size].mean()
        for i in range(n_windows)
    ])

def extract_frame_landmarks(row):
    xs = [float(row[f"X{i}"]) for i in range(68)]
    ys = [float(row[f"Y{i}"]) for i in range(68)]
    zs = [float(row[f"Z{i}"]) for i in range(68)]
    return np.stack([xs, ys, zs], axis=1).astype(np.float32)

def process_participant(clnf_path, covarep_path, edge_index, window_size=30):
    df_clnf      = load_clnf_3d(clnf_path)
    df_audio     = load_covarep(covarep_path)
    df_audio_agg = aggregate_covarep(df_audio, window_size)

    audio_step_s = (window_size * 10) / 1000.0
    df_audio_agg["timestamp"] = [i * audio_step_s for i in range(len(df_audio_agg))]
    n_audio = len(df_audio_agg)

    graphs = []
    for _, row in df_clnf.iterrows():
        if int(row["success"]) == 0:
            continue

        clnf_ts   = float(row["timestamp"])
        audio_idx = min(int(clnf_ts / audio_step_s), n_audio - 1)

        landmarks = extract_frame_landmarks(row)
        x         = torch.tensor(landmarks, dtype=torch.float)
        audio     = torch.tensor(
            df_audio_agg.iloc[audio_idx].drop("timestamp").values.astype(np.float32),
            dtype=torch.float
        )
        graphs.append({
            "x"          : x,
            "edge_index" : edge_index,
            "audio"      : audio,
        })
    return graphs

# --- Fetch ZIP list ------
soup = BeautifulSoup(requests.get(BASE_URL).text, "html.parser")
zip_files = sorted(set(
    l.get("href") for l in soup.find_all("a")
    if l.get("href", "").endswith("_P.zip")
    and int(l.get("href").split("_")[0]) in allowed_ids
))
print(f"ZIPs found: {len(zip_files)}")

# --- Main loop ---------------
skipped, processed, failed = [], [], []

for zip_name in zip_files:
    pid       = zip_name.replace(".zip", "")
    save_path = os.path.join(SAVE_DIR, f"{pid}.pt")

    # Already processed  skip
    if os.path.exists(save_path):
        print(f"[SKIP] {pid}  graph already exists")
        skipped.append(pid)
        continue

    # No label  skip
    if pid not in df_labels.index:
        print(f"[SKIP] {pid}  no label found")
        skipped.append(pid)
        continue

    label     = int(df_labels.loc[pid, "PHQ8_Class"])
    phq_score = int(df_labels.loc[pid, "PHQ8_Score"])

    print(f"\n--- {pid} ---")

    # 1. Download ZIP
    try:
        r = requests.get(BASE_URL + zip_name, timeout=120)
        r.raise_for_status()
    except Exception as e:
        print(f"  [ERROR] Download: {e}")
        failed.append(pid)
        continue

    # 2. Extract only CLNF and COVAREP to /tmp
    clnf_path    = None
    covarep_path = None
    extracted    = []

    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            for info in z.infolist():
                fname = os.path.basename(info.filename)
                if "CLNF_features3D" in fname or "COVAREP" in fname:
                    out_path = os.path.join(TMP_DIR, fname)
                    with z.open(info) as fsrc, open(out_path, "wb") as fdst:
                        fdst.write(fsrc.read())
                    extracted.append(out_path)
                    print(f"  [EXTRACT] {fname}")
                    if "CLNF_features3D" in fname:
                        clnf_path = out_path
                    elif "COVAREP" in fname:
                        covarep_path = out_path
    except Exception as e:
        print(f"  [ERROR] Extraction: {e}")
        failed.append(pid)
        continue

    if not clnf_path or not covarep_path:
        print(f"  [WARN] Missing CLNF or COVAREP in ZIP")
        for f in extracted: os.remove(f)
        failed.append(pid)
        continue

    # 3. Process  graphs
    try:
        graphs = process_participant(clnf_path, covarep_path, edge_index)

        if not graphs:
            print(f"  [WARN] No valid frames")
            failed.append(pid)
        else:
            y_tensor   = torch.tensor(label,     dtype=torch.long)
            phq_tensor = torch.tensor(phq_score, dtype=torch.float)
            for g in graphs:
                g["y"]         = y_tensor
                g["phq_score"] = phq_tensor

            torch.save(graphs, save_path)
            print(f"  [OK] {len(graphs)} frames  {save_path}")
            print(f"       Size: {os.path.getsize(save_path)/1024**2:.1f} MB")
            processed.append(pid)

    except Exception as e:
        print(f"  [ERROR] Processing: {e}")
        failed.append(pid)

    # 4. Always clean up /tmp regardless of success/failure
    for f in extracted:
        if os.path.exists(f):
            os.remove(f)
    print(f"  [DEL] Temp files cleaned")

    time.sleep(1)

# --- Summary ------------------
print(f"\n{'---'*45}")
print(f"Processed : {len(processed)}")
print(f"Skipped   : {len(skipped)}")
print(f"Failed    : {len(failed)}")
if failed:
    print(f"  {failed}")


## 5. Start Here When Preprocessed Files Already Exist

If the `.pt` graph and embedding files are already available, execution can usually start from the dataset/topology cells below.


In [ ]:

# ============================================================
# BLOCK 2: Topolog Facial (68 landmarks, bidireccional)
# ============================================================
FACIAL_EDGES = [
    # Jaw
    (0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),
    (8,9),(9,10),(10,11),(11,12),(12,13),(13,14),(14,15),(15,16),
    # Left eyebrow
    (17,18),(18,19),(19,20),(20,21),
    # Right eyebrow
    (22,23),(23,24),(24,25),(25,26),
    # Nose
    (27,28),(28,29),(29,30),
    (30,31),(31,32),(32,33),(33,34),(34,35),
    # Left eye
    (36,37),(37,38),(38,39),(39,40),(40,41),(41,36),
    # Right eye
    (42,43),(43,44),(44,45),(45,46),(46,47),(47,42),
    # Mouth outer
    (48,49),(49,50),(50,51),(51,52),(52,53),(53,54),
    (54,55),(55,56),(56,57),(57,58),(58,59),(59,48),
]

# Bidireccional (requerido por PyG)
edges_both = FACIAL_EDGES + [(b, a) for a, b in FACIAL_EDGES]
edge_index = torch.tensor(edges_both, dtype=torch.long).t().contiguous()

print(f"Edges totales (bidireccional): {edge_index.shape[1]}")

_edge_index_cache = {}

def get_expanded_edge_index(edge_index, B, W, N, device):
    key = (B * W, N, str(device))
    if key not in _edge_index_cache:
        offsets  = torch.arange(B * W, device=device) * N
        expanded = edge_index.to(device).unsqueeze(0) + offsets.view(-1, 1, 1)
        _edge_index_cache[key] = expanded.permute(1, 0, 2).reshape(2, -1)
    return _edge_index_cache[key]

def get_batch_vec(B, W, N, device):
    key = (B * W, N, str(device), "bv")
    if key not in _edge_index_cache:
        _edge_index_cache[key] = torch.arange(
            B * W, device=device
        ).repeat_interleave(N)
    return _edge_index_cache[key]


## 6. Facial Graph Topology and Dataset Definition

The fixed facial landmark graph is defined here, followed by dataset indexing, participant splits, and feature normalization.


In [ ]:
import os
import torch
import numpy as np
from collections import Counter
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedShuffleSplit

# ============================================================
# Participant-level repeated stratified splits
# ============================================================
train_ids = [
    303, 304, 305, 310, 312, 313, 315, 316, 317, 318, 319, 320, 321,
    322, 324, 325, 326, 327, 328, 330, 333, 336, 338, 339, 340, 341,
    343, 344, 345, 347, 348, 350, 351, 352, 353, 355, 356, 357, 358,
    360, 362, 363, 364, 366, 368, 369, 370, 371, 372, 374, 375, 376,
    379, 380, 383, 385, 386, 391, 392, 393, 397, 400, 401, 402, 409,
    412, 414, 415, 416, 419, 423, 425, 426, 427, 428, 429, 430, 433,
    434, 437, 441, 443, 444, 445, 446, 447, 448, 449, 454, 455, 456,
    457, 459, 463, 464, 468, 471, 473, 474, 475, 478, 479, 485, 486,
    487, 488, 491,
]

dev_ids = [
    302, 307, 331, 335, 346, 367, 377, 381, 382, 388, 389, 390, 395,
    403, 404, 406, 413, 417, 418, 420, 422, 436, 439, 440, 451, 458,
    472, 476, 477, 482, 483, 484, 489, 490, 492,
]

test_ids = dev_ids  # Keep official dev/test intact.
N_REPEATED_SPLITS = 5
VAL_SIZE = 0.15

def get_pid_label(pid, graphs_dir=GRAPHS_DIR):
    pid_key = f"{pid}_P"
    if "df_labels" in globals() and pid_key in df_labels.index:
        return int(df_labels.loc[pid_key, "PHQ8_Class"])
    graph_path = os.path.join(graphs_dir, f"{pid_key}.pt")
    graphs = torch.load(graph_path, weights_only=True)
    return int(graphs[0]["y"].item())

participant_labels = {pid: get_pid_label(pid) for pid in train_ids}
y_train_participants = np.array([participant_labels[pid] for pid in train_ids])

print("--- Participant distribution in TRAIN pool ---")
for cls, name in CLASS_NAMES.items():
    print(f"  {name:25s}: {sum(y_train_participants == cls):3d} participants")

min_class_count = min(Counter(y_train_participants).values())
if min_class_count >= 2:
    splitter = StratifiedShuffleSplit(
        n_splits=N_REPEATED_SPLITS,
        test_size=VAL_SIZE,
        random_state=SEED,
    )
    split_iter = splitter.split(np.array(train_ids), y_train_participants)
else:
    print("[WARN] At least one class has <2 participants; using non-stratified repeated splits.")
    rng = np.random.default_rng(SEED)
    split_iter = []
    for _ in range(N_REPEATED_SPLITS):
        perm = rng.permutation(len(train_ids))
        n_val = max(1, int(round(len(train_ids) * VAL_SIZE)))
        val_idx = perm[:n_val]
        train_idx = perm[n_val:]
        split_iter.append((train_idx, val_idx))

CV_SPLITS = []
for fold, (tr_idx, va_idx) in enumerate(split_iter):
    tr = [train_ids[i] for i in tr_idx]
    va = [train_ids[i] for i in va_idx]
    CV_SPLITS.append({"fold": fold, "train": tr, "val": va})
    print(f"\nFold {fold}: train={len(tr)} val={len(va)}")
    for split_name, pids in [("train", tr), ("val", va)]:
        counts = Counter(participant_labels[pid] for pid in pids)
        msg = " | ".join(f"{CLASS_NAMES[c]}={counts.get(c, 0)}" for c in CLASS_NAMES)
        print(f"  {split_name:5s}: {msg}")

ACTIVE_FOLD = 0
train_final = CV_SPLITS[ACTIVE_FOLD]["train"]
val_ids = CV_SPLITS[ACTIVE_FOLD]["val"]

print(f"\nActive fold: {ACTIVE_FOLD}")
print(f"Train : {len(train_final)} participants")
print(f"Val   : {len(val_ids)} participants")
print(f"Test  : {len(test_ids)} participants")

# ============================================================
# Normalisation stats, computed on train participants only
# ============================================================
def compute_norm_stats(pids, graphs_dir):
    all_x, all_audio = [], []

    for pid in pids:
        path = os.path.join(graphs_dir, f"{pid}_P.pt")
        if not os.path.exists(path):
            continue
        graphs = torch.load(path, weights_only=True)
        if not graphs:
            continue

        x_t = torch.stack([g["x"] for g in graphs])
        aud_t = torch.stack([g["audio"] for g in graphs])
        x_t = torch.nan_to_num(x_t, nan=0.0, posinf=0.0, neginf=0.0)
        aud_t = torch.nan_to_num(aud_t, nan=0.0, posinf=0.0, neginf=0.0)

        all_x.append(x_t)
        all_audio.append(aud_t)

    all_x = torch.cat(all_x, dim=0)
    all_audio = torch.cat(all_audio, dim=0)

    x_mean = all_x.mean(dim=(0, 1), keepdim=True)
    x_std = all_x.std(dim=(0, 1), keepdim=True)
    x_std[x_std < 1e-5] = 1.0

    a_mean = all_audio.mean(dim=0, keepdim=True)
    a_std = all_audio.std(dim=0, keepdim=True)
    a_std[a_std < 1e-5] = 1.0

    return x_mean, x_std, a_mean, a_std


class GraphAugmentor:
    def __init__(self, node_noise_std=0.01, node_drop_prob=0.05,
                 edge_drop_prob=0.05, flip_prob=0.5, enabled=True):
        self.node_noise_std = node_noise_std
        self.node_drop_prob = node_drop_prob
        self.edge_drop_prob = edge_drop_prob
        self.flip_prob = flip_prob
        self.enabled = enabled

    def __call__(self, x_face, edge_index):
        if not self.enabled:
            return x_face, edge_index

        if self.node_noise_std > 0:
            x_face = x_face + torch.randn_like(x_face) * self.node_noise_std

        if self.node_drop_prob > 0:
            mask = torch.rand(x_face.shape[1], device=x_face.device) > self.node_drop_prob
            x_face = x_face * mask.unsqueeze(0).unsqueeze(-1)

        if self.edge_drop_prob > 0:
            mask = torch.rand(edge_index.shape[1], device=edge_index.device) > self.edge_drop_prob
            edge_index = edge_index[:, mask]

        if torch.rand(1).item() < self.flip_prob:
            x_face = x_face.clone()
            x_face[..., 0] = -x_face[..., 0]

        return x_face, edge_index

    def augment_audio(self, x_audio):
        if not self.enabled:
            return x_audio
        return x_audio + torch.randn_like(x_audio) * self.node_noise_std


class DAICDataset(Dataset):
    def __init__(
        self,
        pids,
        graphs_dir,
        wav2vec_dir,
        edge_index,
        x_mean, x_std,
        a_mean, a_std,
        window_frames=10,
        stride_frames=10,
        augment=False,
    ):
        self.graphs_dir = graphs_dir
        self.wav2vec_dir = wav2vec_dir
        self.edge_index = edge_index
        self.x_mean = x_mean
        self.x_std = x_std
        self.a_mean = a_mean
        self.a_std = a_std
        self.window_frames = window_frames
        self.stride_frames = stride_frames
        self.augment = augment
        self.augmentor = GraphAugmentor(enabled=augment)

        self.samples = []
        self.pid_cache = {}

        clnf_step_s = 1 / 30
        wav2vec_win_s = 0.3

        for pid in pids:
            graph_path = os.path.join(graphs_dir, f"{pid}_P.pt")
            wav2vec_path = os.path.join(wav2vec_dir, f"{pid}_P.pt")

            if not os.path.exists(graph_path):
                print(f"[MISSING] graph  : {pid}_P.pt")
                continue
            if not os.path.exists(wav2vec_path):
                print(f"[MISSING] wav2vec: {pid}_P.pt")
                continue

            graphs = torch.load(graph_path, weights_only=True)
            if not graphs:
                continue

            n_frames = len(graphs)
            if n_frames < window_frames:
                continue

            x_t = torch.stack([g["x"] for g in graphs])
            aud_t = torch.stack([g["audio"] for g in graphs])
            x_t = torch.nan_to_num(x_t, nan=0.0, posinf=0.0, neginf=0.0)
            aud_t = torch.nan_to_num(aud_t, nan=0.0, posinf=0.0, neginf=0.0)

            x_t = (x_t - x_mean) / x_std
            aud_t = (aud_t - a_mean) / a_std

            wav2vec = torch.load(wav2vec_path, weights_only=True)
            if isinstance(wav2vec, dict):
                wav2vec = wav2vec["embeddings"]
            wav2vec = torch.nan_to_num(wav2vec, nan=0.0, posinf=0.0, neginf=0.0)

            y = int(graphs[0]["y"].item())
            n_emb = wav2vec.shape[0]

            self.pid_cache[pid] = {
                "x": x_t,
                "audio": aud_t,
                "wav2vec": wav2vec,
                "y": y,
                "n_emb": n_emb,
                "clnf_step": clnf_step_s,
                "wav2vec_win": wav2vec_win_s,
            }

            for start in range(0, n_frames - window_frames + 1, stride_frames):
                self.samples.append((pid, start))

        print(f"Windows indexed: {len(self.samples)} (augment={augment})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, start = self.samples[idx]
        cache = self.pid_cache[pid]
        end = start + self.window_frames

        x_face = cache["x"][start:end].clone()
        x_audio = cache["audio"][start:end].clone()

        n_emb = cache["n_emb"]
        indices = [
            min(int((start + j) * cache["clnf_step"] / cache["wav2vec_win"]), n_emb - 1)
            for j in range(self.window_frames)
        ]
        x_audio_w2v = cache["wav2vec"][indices]

        ei = self.edge_index.clone()
        x_face, ei = self.augmentor(x_face, ei)
        x_audio = self.augmentor.augment_audio(x_audio)
        x_audio_w2v = self.augmentor.augment_audio(x_audio_w2v)

        return {
            "x_face": x_face,
            "x_audio": x_audio,
            "x_audio_w2v": x_audio_w2v,
            "edge_index": ei,
            "y": torch.tensor(cache["y"], dtype=torch.long),
            "pid": pid,
        }


x_mean, x_std, a_mean, a_std = compute_norm_stats(train_final, GRAPHS_DIR)
print("Normalisation stats computed.")

train_dataset = DAICDataset(train_final, GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                            x_mean, x_std, a_mean, a_std,
                            window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                            augment=True)
val_dataset = DAICDataset(val_ids, GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                          x_mean, x_std, a_mean, a_std,
                          window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                          augment=False)
test_dataset = DAICDataset(test_ids, GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                           x_mean, x_std, a_mean, a_std,
                           window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                           augment=False)

print(f"Train windows : {len(train_dataset)}")
print(f"Val   windows : {len(val_dataset)}")
print(f"Test  windows : {len(test_dataset)}")


## 7. Live Metric Visualization

Optional callback utilities for monitoring training curves during notebook execution.


In [ ]:

class LiveMetricsLogger(Callback):
    """
    Callback de PyTorch Lightning que muestra en tiempo real

    las curvas de loss, accuracy y F1 para train y val.
    Se actualiza al final de cada epoch.
    """

    def __init__(self):
        self.train_loss, self.val_loss = [], []
        self.train_acc,  self.val_acc  = [], []
        self.train_f1,   self.val_f1   = [], []
        self.epochs = []

    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        self.epochs.append(trainer.current_epoch + 1)
        self.train_loss.append(metrics.get("train_loss", torch.tensor(0)).item())
        self.train_acc.append(metrics.get("train_acc",  torch.tensor(0)).item())
        self.train_f1.append(metrics.get("train_f1",   torch.tensor(0)).item())

    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics

        # Ignorar sanity check (epoch 0)
        if trainer.current_epoch == 0 and trainer.global_step == 0:
            return

        self.val_loss.append(metrics.get("val_loss", torch.tensor(0)).item())
        self.val_acc.append(metrics.get("val_acc",   torch.tensor(0)).item())
        self.val_f1.append(metrics.get("val_f1",     torch.tensor(0)).item())

        self._plot()

    def _plot(self):
        n = min(len(self.train_loss), len(self.val_loss))
        if n == 0:
            return

        epochs = self.epochs[:n]

        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle(f"Epoch {epochs[-1]}", fontsize=13, fontweight="bold")

        # Loss
        axes[0].plot(epochs, self.train_loss[:n], label="Train", marker="o", markersize=3)
        axes[0].plot(epochs, self.val_loss[:n],   label="Val",   marker="o", markersize=3)
        axes[0].set_title("Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Accuracy
        axes[1].plot(epochs, self.train_acc[:n], label="Train", marker="o", markersize=3)
        axes[1].plot(epochs, self.val_acc[:n],   label="Val",   marker="o", markersize=3)
        axes[1].set_title("Accuracy")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylim(0, 1)
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        # F1
        axes[2].plot(epochs, self.train_f1[:n], label="Train", marker="o", markersize=3)
        axes[2].plot(epochs, self.val_f1[:n],   label="Val",   marker="o", markersize=3)
        axes[2].set_title("F1 Score (macro)")
        axes[2].set_xlabel("Epoch")
        axes[2].set_ylim(0, 1)
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        display(fig)
        plt.close(fig)


## 8. Participant-Aware Sampling

Sampler utilities used to keep training batches aware of participant-level grouping and class imbalance.


In [ ]:
from torch.utils.data import Sampler, WeightedRandomSampler

class ParticipantGroupedSampler(Sampler):
    """
    Kept only for ablation/debugging. The main training loaders below use mixed
    batches, because grouped participant batches inflate batch-level F1 and can
    destabilise BatchNorm.
    """
    def __init__(self, dataset, shuffle=True):
        self.dataset = dataset
        self.shuffle = shuffle
        self.pid_to_indices = defaultdict(list)
        for idx, (pid, _) in enumerate(dataset.samples):
            self.pid_to_indices[pid].append(idx)
        self.pids = list(self.pid_to_indices.keys())

    def __iter__(self):
        pids = self.pids.copy()
        if self.shuffle:
            random.shuffle(pids)
        for pid in pids:
            indices = self.pid_to_indices[pid].copy()
            if self.shuffle:
                random.shuffle(indices)
            yield from indices

    def __len__(self):
        return len(self.dataset)


def make_window_sampler_from_participant_weights(dataset, participant_weight_by_pid):
    sample_weights = [
        float(participant_weight_by_pid.get(pid, 1.0))
        for pid, _ in dataset.samples
    ]
    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )


## 9. DataLoaders

DataLoader construction, collation, and split summaries for train, validation, and external test evaluation.


In [ ]:
from torch.utils.data import DataLoader
from collections import Counter
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def collate_fn(batch):
    return {
        "x_face": torch.stack([b["x_face"] for b in batch]),
        "x_audio": torch.stack([b["x_audio"] for b in batch]),
        "x_audio_w2v": torch.stack([b["x_audio_w2v"] for b in batch]),
        "edge_index": batch[0]["edge_index"],
        "y": torch.stack([b["y"] for b in batch]),
        "pid": [b["pid"] for b in batch],
    }


def participant_class_counts(dataset):
    return Counter(cache["y"] for cache in dataset.pid_cache.values())


def make_loaders(train_pids, val_pids, test_pids,
                 graphs_dir, wav2vec_dir, edge_index,
                 x_mean, x_std, a_mean, a_std,
                 batch_size=BATCH_SIZE,
                 use_balanced_sampler=True):

    train_ds = DAICDataset(
        pids=train_pids, graphs_dir=graphs_dir, wav2vec_dir=wav2vec_dir,
        edge_index=edge_index, x_mean=x_mean, x_std=x_std,
        a_mean=a_mean, a_std=a_std,
        window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
        augment=True,
    )
    val_ds = DAICDataset(
        pids=val_pids, graphs_dir=graphs_dir, wav2vec_dir=wav2vec_dir,
        edge_index=edge_index, x_mean=x_mean, x_std=x_std,
        a_mean=a_mean, a_std=a_std,
        window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
        augment=False,
    )
    test_ds = DAICDataset(
        pids=test_pids, graphs_dir=graphs_dir, wav2vec_dir=wav2vec_dir,
        edge_index=edge_index, x_mean=x_mean, x_std=x_std,
        a_mean=a_mean, a_std=a_std,
        window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
        augment=False,
    )

    print("\n--- Participant distribution ---")
    for split_name, ds in [("TRAIN", train_ds), ("VAL", val_ds), ("TEST", test_ds)]:
        counts = participant_class_counts(ds)
        print(f"{split_name}:")
        for cls, name in CLASS_NAMES.items():
            print(f"  {name:25s}: {counts.get(cls, 0):3d} participants")

    sampler = None
    shuffle = True
    if use_balanced_sampler:
        class_counts = participant_class_counts(train_ds)
        participant_weight_by_pid = {}
        for pid, cache in train_ds.pid_cache.items():
            participant_weight_by_pid[pid] = 1.0 / max(class_counts[cache["y"]], 1)
        sampler = make_window_sampler_from_participant_weights(train_ds, participant_weight_by_pid)
        shuffle = False

    train_loader = DataLoader(
        train_ds, batch_size=batch_size,
        shuffle=shuffle, sampler=sampler,
        collate_fn=collate_fn,
        num_workers=4, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=4, pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=4, pin_memory=True,
    )

    print(f"\n[OK] Train : {len(train_ds)} windows (augment=True)")
    print(f"     Val   : {len(val_ds)} windows (augment=False)")
    print(f"     Test  : {len(test_ds)} windows (augment=False)")
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(
    train_pids=train_final,
    val_pids=val_ids,
    test_pids=test_ids,
    graphs_dir=GRAPHS_DIR,
    wav2vec_dir=WAV2VEC_DIR,
    edge_index=edge_index,
    x_mean=x_mean, x_std=x_std,
    a_mean=a_mean, a_std=a_std,
    batch_size=BATCH_SIZE,
)


## 10. CORN Model Definitions

This section defines the ordinal CORN target transformation, participant-level aggregation utilities, and the Baseline, COVAREP, and Wav2Vec CORN models.


In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch_geometric.nn import GCNConv, SAGEConv, global_mean_pool
from sklearn.metrics import f1_score, accuracy_score
from collections import defaultdict


def get_expanded_edge_index(edge_index, n_graphs, n_edges_per_graph, n_nodes, device):
    offsets = torch.arange(n_graphs, device=device) * n_nodes
    ei = edge_index.to(device).unsqueeze(0).expand(n_graphs, -1, -1)
    ei = ei + offsets.view(n_graphs, 1, 1)
    return ei.reshape(2, -1)


def corn_targets(y, num_classes):
    """Convert integer ordinal labels into CORN binary threshold targets."""
    thresholds = torch.arange(num_classes - 1, device=y.device).view(1, -1)
    return (y.view(-1, 1) > thresholds).float()


def corn_logits_to_classes(logits, threshold=0.5):
    """Convert CORN threshold logits into ordinal class predictions."""
    probs = torch.sigmoid(logits)
    return (probs > threshold).sum(dim=1).long()


def participant_predictions_from_logits(logits_list, pids, labels, class_bias=None):
    by_pid_logits = defaultdict(list)
    by_pid_label = {}
    for logits, pid, label in zip(logits_list, pids, labels):
        by_pid_logits[pid].append(torch.as_tensor(logits).float())
        by_pid_label[pid] = int(label)

    preds, labs = [], []
    for pid in by_pid_logits:
        mean_logits = torch.stack(by_pid_logits[pid]).mean(dim=0)
        if class_bias is not None:
            mean_logits = mean_logits + torch.as_tensor(class_bias).float()
        preds.append(int(corn_logits_to_classes(mean_logits.unsqueeze(0))[0].item()))
        labs.append(by_pid_label[pid])
    return np.array(preds), np.array(labs)


class CORNLightningMixin:
    """Shared PyTorch Lightning utilities for CORN loss, metric logging, and ordinal prediction."""
    def _init_corn(self, num_classes):
        self.num_classes = num_classes
        self.register_buffer("corn_pos_weight", torch.ones(num_classes - 1))

    def _corn_loss(self, logits, y):
        targets = corn_targets(y, self.num_classes)
        return F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=self.corn_pos_weight,
        )

    def _start_epoch_buffers(self, stage):
        setattr(self, f"{stage}_preds_epoch", [])
        setattr(self, f"{stage}_labels_epoch", [])
        setattr(self, f"{stage}_logits_epoch", [])
        setattr(self, f"{stage}_pids_epoch", [])

    def on_train_epoch_start(self):
        self._start_epoch_buffers("train")

    def on_validation_epoch_start(self):
        self._start_epoch_buffers("val")

    def _log_epoch_metrics(self, stage):
        labels = np.array(getattr(self, f"{stage}_labels_epoch"))
        preds = np.array(getattr(self, f"{stage}_preds_epoch"))
        if len(labels) == 0:
            return

        window_f1 = f1_score(labels, preds, average="macro", zero_division=0)
        window_acc = accuracy_score(labels, preds)
        self.log(f"{stage}_window_macro_f1", window_f1, prog_bar=(stage == "val"))
        self.log(f"{stage}_window_acc", window_acc, prog_bar=False)

        logits = getattr(self, f"{stage}_logits_epoch")
        pids = getattr(self, f"{stage}_pids_epoch")
        part_preds, part_labels = participant_predictions_from_logits(logits, pids, labels)
        part_f1 = f1_score(part_labels, part_preds, average="macro", zero_division=0)
        part_acc = accuracy_score(part_labels, part_preds)
        self.log(f"{stage}_participant_macro_f1", part_f1, prog_bar=(stage == "val"))
        self.log(f"{stage}_participant_acc", part_acc, prog_bar=False)

    def on_train_epoch_end(self):
        self._log_epoch_metrics("train")

    def on_validation_epoch_end(self):
        self._log_epoch_metrics("val")

    def _shared_corn_step(self, batch, stage, audio_key):
        x_face = batch["x_face"].to(self.device)
        x_audio = batch[audio_key].to(self.device)
        edge_index = batch["edge_index"].to(self.device)
        y = batch["y"].to(self.device)

        logits = self(x_face, x_audio, edge_index)
        loss = self._corn_loss(logits, y)
        preds = corn_logits_to_classes(logits)

        bs = y.shape[0]
        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=bs)

        getattr(self, f"{stage}_preds_epoch").extend(preds.detach().cpu().numpy().tolist())
        getattr(self, f"{stage}_labels_epoch").extend(y.detach().cpu().numpy().tolist())
        getattr(self, f"{stage}_logits_epoch").extend(logits.detach().cpu())
        getattr(self, f"{stage}_pids_epoch").extend(batch["pid"])
        return loss


class BaselineCORNModel(CORNLightningMixin, pl.LightningModule):
    """Baseline multimodal CORN model using facial GCN features, acoustic features, and a temporal GRU."""
    def __init__(self, node_in=3, audio_in=74, gcn_hidden=32,
                 mlp_hidden=32, gru_hidden=64, num_classes=5,
                 dropout=0.5, lr=3e-4, weight_decay=0.01):
        super().__init__()
        self.save_hyperparameters()
        self._init_corn(num_classes)

        self.gcn1 = GCNConv(node_in, gcn_hidden)
        self.bn_g1 = nn.BatchNorm1d(gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)
        self.bn_face = nn.BatchNorm1d(gcn_hidden)

        self.audio_mlp = nn.Sequential(
            nn.Linear(audio_in, mlp_hidden * 2),
            nn.BatchNorm1d(mlp_hidden * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden * 2, mlp_hidden),
            nn.BatchNorm1d(mlp_hidden),
            nn.ReLU(),
        )
        self.gru = nn.GRU(gcn_hidden + mlp_hidden, gru_hidden, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden // 2, num_classes - 1),
        )

    def _encode_faces_gcn(self, x_face, edge_index):
        B, W, N, C = x_face.shape
        x_flat = x_face.reshape(B * W * N, C)
        batch_vec = torch.arange(B * W, device=x_face.device).repeat_interleave(N)
        ei_flat = get_expanded_edge_index(edge_index, B * W, 1, N, x_face.device)
        h = F.relu(self.bn_g1(self.gcn1(x_flat, ei_flat)))
        h = F.relu(self.bn_face(self.gcn2(h, ei_flat)))
        return global_mean_pool(h, batch_vec).view(B, W, -1)

    def forward(self, x_face, x_audio, edge_index):
        B, W = x_face.shape[:2]
        h_face = self._encode_faces_gcn(x_face, edge_index)
        h_audio = self.audio_mlp(x_audio.reshape(B * W, -1)).view(B, W, -1)
        h = torch.cat([h_face, h_audio], dim=-1)
        self.gru.flatten_parameters()
        _, h_n = self.gru(h)
        return self.classifier(h_n[-1])

    def training_step(self, batch, batch_idx):
        return self._shared_corn_step(batch, "train", "x_audio")

    def validation_step(self, batch, batch_idx):
        return self._shared_corn_step(batch, "val", "x_audio")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "monitor": "val_participant_macro_f1", "interval": "epoch"}}


class CovarepCORNModel(CORNLightningMixin, pl.LightningModule):
    """COVAREP-focused CORN model using GraphSAGE facial encoding and acoustic temporal modeling."""
    def __init__(self, node_in=3, audio_in=74, sage_hidden=32,
                 mlp_hidden=32, gru_hidden=64, dropout=0.5,
                 num_classes=5, lr=3e-4, weight_decay=0.01):
        super().__init__()
        self.save_hyperparameters()
        self._init_corn(num_classes)

        self.sage1 = SAGEConv(node_in, sage_hidden * 2)
        self.bn_s1 = nn.BatchNorm1d(sage_hidden * 2)
        self.sage2 = SAGEConv(sage_hidden * 2, sage_hidden)
        self.bn_face = nn.BatchNorm1d(sage_hidden)
        self.audio_mlp = nn.Sequential(
            nn.Linear(audio_in, mlp_hidden * 2),
            nn.BatchNorm1d(mlp_hidden * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden * 2, mlp_hidden),
            nn.BatchNorm1d(mlp_hidden),
            nn.ReLU(),
        )
        self.gru = nn.GRU(sage_hidden + mlp_hidden, gru_hidden, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden // 2, num_classes - 1),
        )

    def _encode_faces_sage(self, x_face, edge_index):
        B, W, N, C = x_face.shape
        x_flat = x_face.reshape(B * W * N, C)
        batch_vec = torch.arange(B * W, device=x_face.device).repeat_interleave(N)
        ei_flat = get_expanded_edge_index(edge_index, B * W, 1, N, x_face.device)
        h = F.relu(self.bn_s1(self.sage1(x_flat, ei_flat)))
        h = F.relu(self.sage2(h, ei_flat))
        h_bn = self.bn_face(global_mean_pool(h, batch_vec))
        return h_bn.view(B, W, -1)

    def forward(self, x_face, x_audio, edge_index):
        B, W = x_face.shape[:2]
        h_face = self._encode_faces_sage(x_face, edge_index)
        h_audio = self.audio_mlp(x_audio.reshape(B * W, -1)).view(B, W, -1)
        h = torch.cat([h_face, h_audio], dim=-1)
        self.gru.flatten_parameters()
        _, h_n = self.gru(h)
        return self.classifier(h_n[-1])

    def training_step(self, batch, batch_idx):
        return self._shared_corn_step(batch, "train", "x_audio")

    def validation_step(self, batch, batch_idx):
        return self._shared_corn_step(batch, "val", "x_audio")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "monitor": "val_participant_macro_f1", "interval": "epoch"}}


class Wav2VecCORNModel(CORNLightningMixin, pl.LightningModule):
    """Wav2Vec-based CORN model combining facial graph features with precomputed Wav2Vec embeddings."""
    def __init__(self, node_in=3, wav2vec_dim=768, audio_proj=64,
                 sage_hidden=32, gru_hidden=128, dropout=0.5,
                 num_classes=5, lr=3e-4, weight_decay=0.01):
        super().__init__()
        self.save_hyperparameters()
        self._init_corn(num_classes)

        self.sage1 = SAGEConv(node_in, sage_hidden * 2)
        self.bn_s1 = nn.BatchNorm1d(sage_hidden * 2)
        self.sage2 = SAGEConv(sage_hidden * 2, sage_hidden)
        self.bn_face = nn.BatchNorm1d(sage_hidden)
        self.audio_proj = nn.Sequential(
            nn.Linear(wav2vec_dim, audio_proj),
            nn.BatchNorm1d(audio_proj),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(sage_hidden + audio_proj, gru_hidden, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden // 2, num_classes - 1),
        )

    def _encode_faces_sage(self, x_face, edge_index):
        B, W, N, C = x_face.shape
        x_flat = x_face.reshape(B * W * N, C)
        batch_vec = torch.arange(B * W, device=x_face.device).repeat_interleave(N)
        ei_flat = get_expanded_edge_index(edge_index, B * W, 1, N, x_face.device)
        h = F.relu(self.bn_s1(self.sage1(x_flat, ei_flat)))
        h = F.relu(self.sage2(h, ei_flat))
        h_bn = self.bn_face(global_mean_pool(h, batch_vec))
        return h_bn.view(B, W, -1)

    def forward(self, x_face, x_audio, edge_index):
        B, W = x_face.shape[:2]
        h_face = self._encode_faces_sage(x_face, edge_index)
        h_audio = self.audio_proj(x_audio.reshape(B * W, -1)).view(B, W, -1)
        h = torch.cat([h_face, h_audio], dim=-1)
        self.gru.flatten_parameters()
        _, h_n = self.gru(h)
        return self.classifier(h_n[-1])

    def training_step(self, batch, batch_idx):
        return self._shared_corn_step(batch, "train", "x_audio_w2v")

    def validation_step(self, batch, batch_idx):
        return self._shared_corn_step(batch, "val", "x_audio_w2v")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "monitor": "val_participant_macro_f1", "interval": "epoch"}}


# Backward-compatible aliases for downstream cells/checkpoints in this notebook.
BaselineGRUModel = BaselineCORNModel
CovarepGRUModel = CovarepCORNModel
Wav2VecGRUModel = Wav2VecCORNModel


## 11. CORN Threshold Weights

Positive weights for the CORN binary thresholds are computed at participant level to mitigate class imbalance without over-counting windows.


In [ ]:
import numpy as np
from collections import Counter


def compute_corn_pos_weights(train_pids, pid_cache, num_classes=5):
    """Compute CORN threshold positive weights from participant-level labels."""
    """CORN threshold weights from one label per participant, not per window."""
    labels = []
    for pid in train_pids:
        if pid in pid_cache:
            labels.append(int(pid_cache[pid]["y"]))

    labels = torch.tensor(labels, dtype=torch.long)
    targets = corn_targets(labels, num_classes)
    positives = targets.sum(dim=0)
    negatives = targets.shape[0] - positives
    pos_weight = negatives / positives.clamp(min=1)
    pos_weight = torch.sqrt(pos_weight)
    pos_weight = pos_weight / pos_weight.mean().clamp(min=1e-8)

    print("Participant labels in train:", Counter(labels.tolist()))
    print("\nCORN positive weights by threshold:")
    for k, w in enumerate(pos_weight.tolist()):
        print(f"  y > {k}: {w:.4f}")
    return pos_weight.float()


corn_pos_weight = compute_corn_pos_weights(
    train_pids=train_final,
    pid_cache=train_dataset.pid_cache,
    num_classes=5,
)

# Backward-compatible name used by the training cell.
class_weights_tensor = corn_pos_weight


## 12. Training

The three CORN models are trained with participant-level validation metrics and checkpointing based on validation participant macro F1.


In [ ]:
# ============================================================
# TRAINING BLOCK: CORN + participant-level early stopping
# ============================================================
import os
import wandb
from pytorch_lightning.loggers import WandbLogger, CSVLogger


def build_model(model_name):
    if model_name == "baseline":
        return BaselineCORNModel(
            node_in=3, audio_in=74, gcn_hidden=32,
            mlp_hidden=32, gru_hidden=64, num_classes=5,
            dropout=0.5, lr=3e-4, weight_decay=0.01,
        )
    if model_name == "covarep":
        return CovarepCORNModel(
            node_in=3, audio_in=74, sage_hidden=32,
            mlp_hidden=32, gru_hidden=64, num_classes=5,
            dropout=0.5, lr=3e-4, weight_decay=0.01,
        )
    if model_name == "wav2vec":
        return Wav2VecCORNModel(
            node_in=3, wav2vec_dim=768, audio_proj=64,
            sage_hidden=32, gru_hidden=128, num_classes=5,
            dropout=0.5, lr=3e-4, weight_decay=0.01,
        )
    raise ValueError(f"Unknown model: {model_name}")


def train_all_models(train_loader, val_loader, corn_pos_weight, fold=ACTIVE_FOLD):
    """Train all CORN model variants for the selected fold and log checkpoints/metrics to W&B."""
    training_group = f"corn_repeated_split_fold{fold}_{wandb.util.generate_id()}"
    results = {}

    for model_name in ["baseline", "covarep", "wav2vec"]:
        run = wandb.init(
            project="depression-gnn",
            group=training_group,
            name=f"{model_name}_fold{fold}",
            config={
                "model": model_name,
                "objective": "CORN",
                "monitor": "val_participant_macro_f1",
                "fold": fold,
                "window_frames": WINDOW_FRAMES,
                "stride_frames": STRIDE_FRAMES,
                "batch_size": BATCH_SIZE,
                "seed": SEED,
                "num_classes": 5,
                "train_participants": len(train_loader.dataset.pid_cache),
                "val_participants": len(val_loader.dataset.pid_cache),
            },
        )

        model = build_model(model_name)
        model.corn_pos_weight = corn_pos_weight.to(model.device)

        run.config.update({
            f"{model_name}_hparams": dict(model.hparams),
            f"{model_name}_n_params": sum(p.numel() for p in model.parameters()),
        })

        print(f"\n[INFO] Training CORN model: {model_name} | fold={fold}")
        checkpoint_callback = pl.callbacks.ModelCheckpoint(
            dirpath=CKPT_DIR,
            monitor="val_participant_macro_f1",
            mode="max",
            save_top_k=1,
            filename=f"corn-{model_name}-fold{fold}-{{epoch:02d}}-pf1{{val_participant_macro_f1:.3f}}",
            auto_insert_metric_name=False,
        )
        early_stopping = pl.callbacks.EarlyStopping(
            monitor="val_participant_macro_f1",
            patience=8,
            mode="max",
            min_delta=0.002,
            verbose=True,
        )

        wandb_logger = WandbLogger(experiment=run, prefix=model_name)
        csv_logger = CSVLogger(save_dir=LOG_DIR, name=f"corn_{model_name}_fold{fold}")
        precision = "16-mixed" if torch.cuda.is_available() else "32"

        trainer = pl.Trainer(
            max_epochs=50,
            accelerator="auto",
            devices=1,
            log_every_n_steps=10,
            logger=[wandb_logger, csv_logger],
            callbacks=[checkpoint_callback, early_stopping],
            gradient_clip_val=1.0,
            precision=precision,
        )
        trainer.fit(model, train_loader, val_loader)

        best_ckpt = checkpoint_callback.best_model_path
        best_f1 = checkpoint_callback.best_model_score

        if best_ckpt and os.path.exists(best_ckpt):
            artifact = wandb.Artifact(
                name=f"corn_{model_name}_fold{fold}_ckpt",
                type="model",
                metadata={"model": model_name, "fold": fold, "val_participant_macro_f1": float(best_f1)},
            )
            artifact.add_file(best_ckpt)
            run.log_artifact(artifact)
            print(f"[WANDB] Artifact logged: {best_ckpt}")

        print(f"\n-- {model_name} fold {fold} --")
        print(f"Best val participant macro F1 : {float(best_f1):.4f}")
        print(f"Checkpoint                    : {best_ckpt}")

        results[model_name] = {
            "model": model,
            "trainer": trainer,
            "ckpt": checkpoint_callback,
            "best_f1": float(best_f1),
            "best_ckpt": best_ckpt,
        }
        wandb.finish()

    return results, training_group


results, training_group = train_all_models(
    train_loader,
    val_loader,
    corn_pos_weight=class_weights_tensor,
    fold=ACTIVE_FOLD,
)

baseline_model = results["baseline"]["model"]
covarep_model = results["covarep"]["model"]
wav2vec_model = results["wav2vec"]["model"]


## 13. Training Curves

Local and W&B-compatible plots for inspecting train/validation loss, accuracy, and macro F1.


In [ ]:
# 3x3 Training curves (local backup) for CORN
from datetime import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
plots_dir = f"/home/arosario/depression-gnn-project/plots/{timestamp}"
os.makedirs(plots_dir, exist_ok=True)

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle("All Models - CORN Training Curves", fontsize=14, fontweight="bold")

for col, (name, key) in enumerate([
    ("Baseline", "baseline"),
    ("COVAREP", "covarep"),
    ("Wav2Vec", "wav2vec"),
]):
    log_root = os.path.join(LOG_DIR, f"corn_{key}_fold{ACTIVE_FOLD}")
    versions = sorted([d for d in os.listdir(log_root) if d.startswith("version_")]) if os.path.exists(log_root) else []
    if not versions:
        print(f"[WARN] No CSV logs found for {key}: {log_root}")
        continue

    log_path = os.path.join(log_root, versions[-1], "metrics.csv")
    df = pd.read_csv(log_path)

    train_loss = df[df["train_loss"].notna()]["train_loss"].values if "train_loss" in df else []
    val_loss = df[df["val_loss"].notna()]["val_loss"].values if "val_loss" in df else []
    train_f1 = df[df["train_participant_macro_f1"].notna()]["train_participant_macro_f1"].values if "train_participant_macro_f1" in df else []
    val_f1 = df[df["val_participant_macro_f1"].notna()]["val_participant_macro_f1"].values if "val_participant_macro_f1" in df else []
    train_wf1 = df[df["train_window_macro_f1"].notna()]["train_window_macro_f1"].values if "train_window_macro_f1" in df else []
    val_wf1 = df[df["val_window_macro_f1"].notna()]["val_window_macro_f1"].values if "val_window_macro_f1" in df else []

    n_loss = min(len(train_loss), len(val_loss))
    if n_loss:
        epochs = range(1, n_loss + 1)
        axes[0][col].plot(epochs, train_loss[:n_loss], label="Train")
        axes[0][col].plot(epochs, val_loss[:n_loss], label="Val")
    axes[0][col].set_title(f"{name} - Loss")
    axes[0][col].set_xlabel("Epoch")
    axes[0][col].legend(); axes[0][col].grid(True, alpha=0.3)

    n_pf1 = min(len(train_f1), len(val_f1))
    if n_pf1:
        epochs = range(1, n_pf1 + 1)
        axes[1][col].plot(epochs, train_f1[:n_pf1], label="Train")
        axes[1][col].plot(epochs, val_f1[:n_pf1], label="Val")
    axes[1][col].set_title(f"{name} - Participant macro F1")
    axes[1][col].set_xlabel("Epoch")
    axes[1][col].set_ylim(0, 1)
    axes[1][col].legend(); axes[1][col].grid(True, alpha=0.3)

    n_wf1 = min(len(train_wf1), len(val_wf1))
    if n_wf1:
        epochs = range(1, n_wf1 + 1)
        axes[2][col].plot(epochs, train_wf1[:n_wf1], label="Train")
        axes[2][col].plot(epochs, val_wf1[:n_wf1], label="Val")
    axes[2][col].set_title(f"{name} - Window macro F1")
    axes[2][col].set_xlabel("Epoch")
    axes[2][col].set_ylim(0, 1)
    axes[2][col].legend(); axes[2][col].grid(True, alpha=0.3)

plt.tight_layout()
save_path = os.path.join(plots_dir, "corn_training_curves.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[SAVED] {save_path}")
if wandb.run is not None:
    wandb.log({"corn_training_curves": wandb.Image(save_path)})


## 14. Evaluation Utilities

Participant-level and window-level evaluation helpers, including classification reports, confusion matrices, bootstrap confidence intervals, and W&B logging.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from collections import Counter, defaultdict
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns


def bootstrap_macro_f1_ci(labels, preds, n_boot=2000, ci=95, seed=SEED):
    labels = np.asarray(labels)
    preds = np.asarray(preds)
    if len(labels) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(labels), len(labels))
        scores.append(f1_score(labels[idx], preds[idx], average="macro", zero_division=0))

    alpha = (100 - ci) / 2
    return (
        float(f1_score(labels, preds, average="macro", zero_division=0)),
        float(np.percentile(scores, alpha)),
        float(np.percentile(scores, 100 - alpha)),
    )


def collect_logits(model, loader, device):
    model.eval()
    all_logits, all_preds, all_labels, all_pids = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            x_face = batch["x_face"].to(device)
            edge_index = batch["edge_index"].to(device)
            y = batch["y"].to(device)

            if isinstance(model, Wav2VecCORNModel):
                x_audio = batch["x_audio_w2v"].to(device)
            else:
                x_audio = batch["x_audio"].to(device)

            logits = model(x_face, x_audio, edge_index)
            preds = corn_logits_to_classes(logits)
            all_logits.extend(logits.cpu())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_pids.extend(batch["pid"])

    return all_logits, np.array(all_preds), np.array(all_labels), all_pids


def aggregate_participant_logits(logits, labels, pids, threshold_bias=None):
    by_pid_logits = defaultdict(list)
    by_pid_label = {}
    for logit, label, pid in zip(logits, labels, pids):
        by_pid_logits[pid].append(torch.as_tensor(logit).float())
        by_pid_label[pid] = int(label)

    part_preds, part_labels, part_pids = [], [], []
    for pid in by_pid_logits:
        mean_logits = torch.stack(by_pid_logits[pid]).mean(dim=0)
        if threshold_bias is not None:
            mean_logits = mean_logits + torch.as_tensor(threshold_bias).float()
        pred = int(corn_logits_to_classes(mean_logits.unsqueeze(0))[0].item())
        part_preds.append(pred)
        part_labels.append(by_pid_label[pid])
        part_pids.append(pid)

    return np.array(part_preds), np.array(part_labels), part_pids


def optimise_corn_threshold_bias(val_logits, val_labels, val_pids, grid=(-1.0, -0.5, 0.0, 0.5, 1.0)):
    """Tune a small CORN logit bias on validation participants only."""
    """Small validation-only calibration over CORN threshold logits."""
    best_bias = np.zeros(4, dtype=np.float32)
    best_f1 = -1.0

    for b0 in grid:
        for b1 in grid:
            for b2 in grid:
                for b3 in grid:
                    bias = np.array([b0, b1, b2, b3], dtype=np.float32)
                    preds, labels, _ = aggregate_participant_logits(val_logits, val_labels, val_pids, bias)
                    score = f1_score(labels, preds, average="macro", zero_division=0)
                    if score > best_f1:
                        best_f1 = score
                        best_bias = bias
    return best_bias, best_f1


def report_to_wandb_table(report):
    rows = []
    for cls_id, cls_name in CLASS_NAMES.items():
        metrics = report.get(cls_name, {})
        rows.append([
            cls_id,
            cls_name,
            metrics.get("precision", 0.0),
            metrics.get("recall", 0.0),
            metrics.get("f1-score", 0.0),
            metrics.get("support", 0),
        ])
    return wandb.Table(
        columns=["class_id", "class", "precision", "recall", "f1", "support"],
        data=rows,
    )


def save_confusion_matrix(labels, preds, title, path):
    cm = confusion_matrix(labels, preds, labels=list(CLASS_NAMES.keys()))
    cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=list(CLASS_NAMES.values()),
        yticklabels=list(CLASS_NAMES.values()),
        ax=axes[0],
    )
    axes[0].set_title("Counts")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")

    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=list(CLASS_NAMES.values()),
        yticklabels=list(CLASS_NAMES.values()),
        ax=axes[1],
    )
    axes[1].set_title("Normalised recall")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def log_eval_to_wandb(run, model_name, loader_name, level, labels, preds, report, cm_path,
                      participant_ci=None, threshold_bias=None):
    prefix = f"{model_name}/{loader_name}/{level}"
    payload = {
        f"{prefix}/accuracy": accuracy_score(labels, preds),
        f"{prefix}/macro_f1": report["macro avg"]["f1-score"],
        f"{prefix}/macro_precision": report["macro avg"]["precision"],
        f"{prefix}/macro_recall": report["macro avg"]["recall"],
        f"{prefix}/weighted_f1": report["weighted avg"]["f1-score"],
        f"{prefix}/per_class": report_to_wandb_table(report),
        f"{prefix}/confusion_matrix": wandb.Image(cm_path),
    }
    if participant_ci is not None:
        f1, lo, hi = participant_ci
        payload[f"{prefix}/macro_f1_bootstrap"] = f1
        payload[f"{prefix}/macro_f1_ci_low"] = lo
        payload[f"{prefix}/macro_f1_ci_high"] = hi
    if threshold_bias is not None:
        payload[f"{model_name}/{loader_name}/threshold_bias"] = wandb.Table(
            columns=["threshold", "bias"],
            data=[[f"y>{i}", float(b)] for i, b in enumerate(threshold_bias)],
        )
    run.log(payload)


def evaluate_model(model, loader, loader_name, model_name, device,
                   plots_dir=None, run=None, training_group=None,
                   threshold_bias=None, n_boot=2000):
    """Evaluate one CORN model at window and participant level."""
    logits, window_preds, window_labels, pids = collect_logits(model, loader, device)
    part_preds, part_labels, part_pids = aggregate_participant_logits(
        logits, window_labels, pids, threshold_bias=threshold_bias
    )

    print(f"\nPrediction distribution - window level ({model_name} - {loader_name}):")
    pred_counts = Counter(window_preds)
    label_counts = Counter(window_labels)
    for cls, name in CLASS_NAMES.items():
        print(f"  {name:25s}: predicted={pred_counts.get(cls, 0):5d}  true={label_counts.get(cls, 0):5d}")

    window_report = classification_report(
        window_labels, window_preds,
        labels=list(CLASS_NAMES.keys()),
        target_names=list(CLASS_NAMES.values()),
        zero_division=0,
        digits=4,
        output_dict=True,
    )
    print(f"\n{model_name} - {loader_name} - WINDOW LEVEL")
    print(classification_report(
        window_labels, window_preds,
        labels=list(CLASS_NAMES.keys()),
        target_names=list(CLASS_NAMES.values()),
        zero_division=0,
        digits=4,
    ))

    print(f"\nPrediction distribution - participant level:")
    pred_counts_p = Counter(part_preds)
    label_counts_p = Counter(part_labels)
    for cls, name in CLASS_NAMES.items():
        print(f"  {name:25s}: predicted={pred_counts_p.get(cls, 0):3d}  true={label_counts_p.get(cls, 0):3d}")

    f1, lo, hi = bootstrap_macro_f1_ci(part_labels, part_preds, n_boot=n_boot)
    participant_report = classification_report(
        part_labels, part_preds,
        labels=list(CLASS_NAMES.keys()),
        target_names=list(CLASS_NAMES.values()),
        zero_division=0,
        digits=4,
        output_dict=True,
    )
    print(f"\n{model_name} - {loader_name} - PARTICIPANT LEVEL ({len(part_labels)} participants)")
    print(classification_report(
        part_labels, part_preds,
        labels=list(CLASS_NAMES.keys()),
        target_names=list(CLASS_NAMES.values()),
        zero_division=0,
        digits=4,
    ))
    print(f"Participant macro F1 bootstrap 95% CI: {f1:.4f} [{lo:.4f}, {hi:.4f}]")

    if run is not None:
        if plots_dir is None:
            plots_dir = os.path.join("/home/arosario/depression-gnn-project/plots", "wandb_eval")
        os.makedirs(plots_dir, exist_ok=True)

        safe_model = str(model_name).replace(" ", "_")
        safe_loader = str(loader_name).replace(" ", "_")
        window_cm_path = os.path.join(plots_dir, f"{safe_model}_{safe_loader}_window_cm.png")
        participant_cm_path = os.path.join(plots_dir, f"{safe_model}_{safe_loader}_participant_cm.png")

        save_confusion_matrix(
            window_labels, window_preds,
            f"{model_name} - {loader_name} - window",
            window_cm_path,
        )
        save_confusion_matrix(
            part_labels, part_preds,
            f"{model_name} - {loader_name} - participant",
            participant_cm_path,
        )

        log_eval_to_wandb(
            run, model_name, loader_name, "window",
            window_labels, window_preds, window_report, window_cm_path,
        )
        log_eval_to_wandb(
            run, model_name, loader_name, "participant",
            part_labels, part_preds, participant_report, participant_cm_path,
            participant_ci=(f1, lo, hi),
            threshold_bias=threshold_bias,
        )

    return {
        "window_preds": window_preds,
        "window_labels": window_labels,
        "participant_preds": part_preds,
        "participant_labels": part_labels,
        "participant_pids": part_pids,
        "participant_macro_f1": f1,
        "participant_macro_f1_ci": (lo, hi),
        "logits": logits,
        "pids": pids,
    }


## 15. Checkpoint-Based Evaluation and Cross-Validation Helpers

Utilities for loading saved CORN checkpoints, calibrating threshold bias on validation data, and evaluating trained models.


In [ ]:
import datetime
import os
from pytorch_lightning.loggers import WandbLogger, CSVLogger


def safe_wandb_init(**kwargs):
    """
    W&B can leave a closed backend service in long notebook sessions. This helper
    closes any active run, tears down the stale service if available, and starts
    a fresh run.
    """
    try:
        if wandb.run is not None:
            wandb.finish()
    except Exception as exc:
        print(f"[WARN] wandb.finish() before init failed: {exc}")

    try:
        wandb.teardown()
    except Exception:
        pass

    kwargs.setdefault("reinit", True)
    try:
        kwargs.setdefault("settings", wandb.Settings(start_method="thread"))
    except Exception:
        pass
    return wandb.init(**kwargs)


def evaluate_all_models(ckpt_paths, val_loader, test_loader,
                        device, plots_dir, training_group):
    """Load all CORN checkpoints and evaluate them on validation and external test data."""
    models_dict = {
        "Baseline": BaselineCORNModel.load_from_checkpoint(
            ckpt_paths["baseline"], strict=False).to(device),
        "COVAREP": CovarepCORNModel.load_from_checkpoint(
            ckpt_paths["covarep"], strict=False).to(device),
        "Wav2Vec": Wav2VecCORNModel.load_from_checkpoint(
            ckpt_paths["wav2vec"], strict=False).to(device),
    }

    run = safe_wandb_init(
        project="depression-gnn",
        group=training_group,
        name="corn_evaluation",
        config={
            "objective": "CORN",
            "aggregation": "mean_participant_logits",
            "calibration": "validation_threshold_bias",
            "bootstrap_repeats": 2000,
            "window_frames": WINDOW_FRAMES,
            "stride_frames": STRIDE_FRAMES,
            "batch_size": BATCH_SIZE,
            "training_group": training_group,
        },
    )

    all_results = {}
    for model_name, model in models_dict.items():
        val_logits, _, val_labels, val_pids = collect_logits(model, val_loader, device)
        threshold_bias, val_calibrated_f1 = optimise_corn_threshold_bias(
            val_logits, val_labels, val_pids
        )
        print(f"\n{model_name} validation threshold bias: {threshold_bias.tolist()} | participant F1={val_calibrated_f1:.4f}")

        all_results[(model_name, "val")] = evaluate_model(
            model, val_loader, "val", model_name, device,
            plots_dir=plots_dir, run=run, training_group=training_group,
            threshold_bias=threshold_bias,
        )
        all_results[(model_name, "test")] = evaluate_model(
            model, test_loader, "test", model_name, device,
            plots_dir=plots_dir, run=run, training_group=training_group,
            threshold_bias=threshold_bias,
        )

    wandb.finish()
    return all_results


def run_repeated_stratified_cv(model_names=("wav2vec",), n_splits=N_REPEATED_SPLITS):
    """
    Repeated stratified validation on the original train pool.
    The official dev/test split remains untouched and is only evaluated after
    selecting a configuration.
    """
    cv_rows = []
    cv_group = f"corn_repeated_cv_{wandb.util.generate_id()}"
    cv_run = safe_wandb_init(
        project="depression-gnn",
        group=cv_group,
        name="repeated_stratified_cv",
        config={
            "objective": "CORN",
            "n_splits": n_splits,
            "model_names": list(model_names),
            "monitor": "val_participant_macro_f1",
            "aggregation": "mean_participant_logits",
            "bootstrap_repeats": 2000,
            "dev_test_untouched": True,
        },
    )
    cv_plots_dir = os.path.join(
        "/home/arosario/depression-gnn-project/plots",
        f"corn_cv_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}",
    )
    os.makedirs(cv_plots_dir, exist_ok=True)

    try:
        for split in CV_SPLITS[:n_splits]:
            fold = split["fold"]
            print(f"\n========== Repeated stratified fold {fold} ==========")
            fold_train = split["train"]
            fold_val = split["val"]
            fold_x_mean, fold_x_std, fold_a_mean, fold_a_std = compute_norm_stats(fold_train, GRAPHS_DIR)
            fold_train_loader, fold_val_loader, _ = make_loaders(
                train_pids=fold_train,
                val_pids=fold_val,
                test_pids=test_ids,
                graphs_dir=GRAPHS_DIR,
                wav2vec_dir=WAV2VEC_DIR,
                edge_index=edge_index,
                x_mean=fold_x_mean, x_std=fold_x_std,
                a_mean=fold_a_mean, a_std=fold_a_std,
                batch_size=BATCH_SIZE,
            )
            fold_pos_weight = compute_corn_pos_weights(
                train_pids=fold_train,
                pid_cache=fold_train_loader.dataset.pid_cache,
                num_classes=5,
            )

            for model_name in model_names:
                model = build_model(model_name)
                model.corn_pos_weight = fold_pos_weight.to(model.device)
                checkpoint_callback = pl.callbacks.ModelCheckpoint(
                    dirpath=CKPT_DIR,
                    monitor="val_participant_macro_f1",
                    mode="max",
                    save_top_k=1,
                    filename=f"cv-corn-{model_name}-fold{fold}-{{epoch:02d}}-pf1{{val_participant_macro_f1:.3f}}",
                    auto_insert_metric_name=False,
                )
                early_stopping = pl.callbacks.EarlyStopping(
                    monitor="val_participant_macro_f1",
                    patience=8,
                    mode="max",
                    min_delta=0.002,
                    verbose=True,
                )
                trainer = pl.Trainer(
                    max_epochs=50,
                    accelerator="auto",
                    devices=1,
                    callbacks=[checkpoint_callback, early_stopping],
                    gradient_clip_val=1.0,
                    precision="16-mixed" if torch.cuda.is_available() else "32",
                    logger=[
                        CSVLogger(save_dir=LOG_DIR, name=f"cv_corn_{model_name}_fold{fold}"),
                        WandbLogger(experiment=cv_run, prefix=f"cv/{model_name}/fold{fold}"),
                    ],
                    log_every_n_steps=10,
                )
                trainer.fit(model, fold_train_loader, fold_val_loader)

                best_model = type(model).load_from_checkpoint(checkpoint_callback.best_model_path, strict=False).to(DEVICE)
                best_model.corn_pos_weight = fold_pos_weight.to(DEVICE)
                val_logits, _, val_labels, val_pids = collect_logits(best_model, fold_val_loader, DEVICE)
                threshold_bias, val_bias_f1 = optimise_corn_threshold_bias(val_logits, val_labels, val_pids)

                eval_res = evaluate_model(
                    best_model,
                    fold_val_loader,
                    f"cv_fold{fold}_val",
                    model_name,
                    DEVICE,
                    plots_dir=cv_plots_dir,
                    run=cv_run,
                    threshold_bias=threshold_bias,
                )
                row = {
                    "fold": fold,
                    "model": model_name,
                    "val_participant_macro_f1": eval_res["participant_macro_f1"],
                    "ci_low": eval_res["participant_macro_f1_ci"][0],
                    "ci_high": eval_res["participant_macro_f1_ci"][1],
                    "threshold_bias_f1": val_bias_f1,
                    "checkpoint": checkpoint_callback.best_model_path,
                }
                cv_rows.append(row)
                cv_run.log({
                    "cv/fold": fold,
                    "cv/model": model_name,
                    f"cv/{model_name}/fold{fold}/participant_macro_f1": row["val_participant_macro_f1"],
                    f"cv/{model_name}/fold{fold}/participant_macro_f1_ci_low": row["ci_low"],
                    f"cv/{model_name}/fold{fold}/participant_macro_f1_ci_high": row["ci_high"],
                    f"cv/{model_name}/fold{fold}/threshold_bias_f1": row["threshold_bias_f1"],
                })

        cv_df = pd.DataFrame(cv_rows)
        display(cv_df)
        print("\nRepeated stratified validation summary:")
        summary = cv_df.groupby("model")["val_participant_macro_f1"].agg(["mean", "std", "count"])
        print(summary)
        cv_run.log({
            "cv/results": wandb.Table(dataframe=cv_df),
            "cv/summary": wandb.Table(dataframe=summary.reset_index()),
        })
        return cv_df
    finally:
        wandb.finish()


In [ ]:
import wandb

api = wandb.Api()
runs = api.runs("alrp-datascience-complutense-university-of-madrid/depression-gnn")

for run in runs:
    print(f"Name: {run.name:20s}  Group: {run.group}  ID: {run.id}")


## 16. Final Checkpoint Evaluation

Use this section to evaluate saved CORN checkpoints and log final validation/test metrics.


In [ ]:
import os
import datetime
import wandb

# Checkpoint-based evaluation from W&B artifacts.
# Run Cell 15 first (evaluation helpers) and Cell 16 second
# (evaluate_all_models / repeated CV definitions), then run this cell.

api = wandb.Api()

ARTIFACT_REFS = {
    "baseline": "alrp-datascience-complutense-university-of-madrid/depression-gnn/corn_baseline_fold0_ckpt:v0",
    "covarep":  "alrp-datascience-complutense-university-of-madrid/depression-gnn/corn_covarep_fold0_ckpt:v0",
    "wav2vec":  "alrp-datascience-complutense-university-of-madrid/depression-gnn/corn_wav2vec_fold0_ckpt:v0",
}

def fetch_ckpt_from_wandb_artifact(
    artifact_ref,
    root="/home/arosario/depression-gnn-project/ckpts/wandb_artifacts",
):
    # Download every artifact into an isolated folder so old .ckpt files from
    # other artifacts cannot be picked up by mistake.
    safe_name = artifact_ref.replace("/", "__").replace(":", "__")
    artifact_root = os.path.join(root, safe_name)

    artifact = api.artifact(artifact_ref, type="model")
    artifact_dir = artifact.download(root=artifact_root)

    ckpt_paths = []
    for dirpath, _, filenames in os.walk(artifact_dir):
        for filename in filenames:
            if filename.endswith(".ckpt"):
                ckpt_paths.append(os.path.join(dirpath, filename))

    if not ckpt_paths:
        raise FileNotFoundError(
            f"No .ckpt file found inside artifact download: {artifact_ref} -> {artifact_dir}"
        )
    if len(ckpt_paths) > 1:
        raise RuntimeError(
            f"Expected exactly one .ckpt for {artifact_ref}, found {len(ckpt_paths)}:\n"
            + "\n".join(ckpt_paths)
        )

    ckpt_path = ckpt_paths[0]
    print(f"[OK] {artifact_ref} -> {ckpt_path}")
    return ckpt_path


ckpt_paths = {
    model_name: fetch_ckpt_from_wandb_artifact(artifact_ref)
    for model_name, artifact_ref in ARTIFACT_REFS.items()
}

training_group = "corn_repeated_split_fold0_c79jjrjj"

if "plots_dir" not in globals():
    plots_dir = os.path.join(
        "/home/arosario/depression-gnn-project/plots",
        f"corn_ckpt_eval_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}",
    )
    os.makedirs(plots_dir, exist_ok=True)

# Reset stale W&B state before evaluate_all_models creates the eval run.
try:
    if wandb.run is not None:
        wandb.finish()
except Exception as exc:
    print(f"[WARN] wandb.finish() before eval failed: {exc}")

try:
    wandb.teardown()
except Exception:
    pass

eval_results = evaluate_all_models(
    ckpt_paths=ckpt_paths,
    val_loader=val_loader,
    test_loader=test_loader,
    device=DEVICE,
    plots_dir=plots_dir,
    training_group=training_group,
)
